# 🏪 Store Sales — Time Series Forecasting System
**Dataset:** Corporación Favorita Grocery Sales (Ecuador, 2013–2017)  
**Target:** Aggregate daily `sales` across all stores and product families  
**Models:** ARIMA · Prophet  
**Horizons:** 7 days · 30 days · 90 days · 6 months

## 1 · Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})

DATA_DIR = ''

train_raw   = pd.read_csv(DATA_DIR + 'train.csv',            parse_dates=['date'])
stores      = pd.read_csv(DATA_DIR + 'stores.csv')
oil         = pd.read_csv(DATA_DIR + 'oil.csv',              parse_dates=['date'])
holidays    = pd.read_csv(DATA_DIR + 'holidays_events.csv',  parse_dates=['date'])
transactions= pd.read_csv(DATA_DIR + 'transactions.csv',     parse_dates=['date'])

print('train shape   :', train_raw.shape)
print('date range    :', train_raw['date'].min().date(), '→', train_raw['date'].max().date())
print('stores        :', train_raw['store_nbr'].nunique())
print('product fams  :', train_raw['family'].nunique())
train_raw.head()

In [ ]:
ts = (
    train_raw
    .groupby('date', as_index=False)['sales']
    .sum()
    .rename(columns={'sales': 'total_sales'})
    .sort_values('date')
    .set_index('date')
)

ts.index.freq = pd.infer_freq(ts.index)
print('Aggregated series shape:', ts.shape)
print('Inferred frequency     :', ts.index.freq)
ts.describe()

## 2 · Time Series Visualisation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), constrained_layout=True)

axes[0].plot(ts.index, ts['total_sales'], color='steelblue', linewidth=0.8, label='Daily Sales')
axes[0].plot(ts['total_sales'].rolling(30).mean(), color='tomato', linewidth=2, label='30-day MA')
peak_idx = ts['total_sales'].idxmax()
dip_idx  = ts['total_sales'].idxmin()
axes[0].annotate(f'Peak\n{peak_idx.date()}', xy=(peak_idx, ts.loc[peak_idx, 'total_sales']),
                 xytext=(peak_idx, ts['total_sales'].max() * 0.88),
                 arrowprops=dict(arrowstyle='->', color='crimson'), color='crimson', fontsize=9)
axes[0].annotate(f'Dip\n{dip_idx.date()}', xy=(dip_idx, ts.loc[dip_idx, 'total_sales']),
                 xytext=(dip_idx, ts['total_sales'].max() * 0.12),
                 arrowprops=dict(arrowstyle='->', color='navy'), color='navy', fontsize=9)
axes[0].set_title('Historical Sales Trend with 30-Day Moving Average', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Sales')
axes[0].legend()

monthly = ts['total_sales'].resample('ME').sum()
axes[1].bar(monthly.index, monthly.values, width=20, color='steelblue', alpha=0.7, label='Monthly Sales')
axes[1].plot(monthly.index, monthly.values, color='tomato', marker='o', markersize=4, linewidth=1.5)
axes[1].set_title('Monthly Aggregated Sales', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Total Sales')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

ts_season = ts.copy()
ts_season['year']    = ts_season.index.year
ts_season['week_of_year'] = ts_season.index.isocalendar().week.astype(int)
pivot = ts_season.pivot_table(index='week_of_year', columns='year', values='total_sales', aggfunc='mean')
for yr in pivot.columns:
    axes[2].plot(pivot.index, pivot[yr], label=str(yr), linewidth=1.2)
axes[2].set_title('Seasonality: Average Sales by Week-of-Year per Year', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Week of Year')
axes[2].set_ylabel('Avg Daily Sales')
axes[2].legend(title='Year', ncol=3)

plt.suptitle('Store Sales — Time Series Overview', fontsize=15, fontweight='bold', y=1.01)
plt.show()

## 3 · Data Preprocessing

In [ ]:
full_idx = pd.date_range(start=ts.index.min(), end=ts.index.max(), freq='D')
missing_dates = full_idx.difference(ts.index)
print(f'Missing timestamps before fill : {len(missing_dates)}')

ts = ts.reindex(full_idx)
ts.index.name = 'date'
ts.index.freq = 'D'

ts['total_sales'] = ts['total_sales'].interpolate(method='time')
print(f'Null values after interpolation: {ts["total_sales"].isna().sum()}')

ts = ts[~ts.index.duplicated(keep='first')]
print(f'Final series shape             : {ts.shape}')

ts_weekly  = ts['total_sales'].resample('W').sum().rename('weekly_sales')
ts_monthly = ts['total_sales'].resample('ME').sum().rename('monthly_sales')

print(f'Weekly  series : {ts_weekly.shape[0]} observations')
print(f'Monthly series : {ts_monthly.shape[0]} observations')

ts.head()

## 4 · Trend, Seasonality & Cyclic Pattern Analysis

In [ ]:
decomp = seasonal_decompose(ts['total_sales'], model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True, constrained_layout=True)
components = [
    (decomp.observed,  'Observed',   'steelblue'),
    (decomp.trend,     'Trend',      'tomato'),
    (decomp.seasonal,  'Seasonality','green'),
    (decomp.resid,     'Residual',   'purple'),
]
for ax, (comp, title, color) in zip(axes, components):
    ax.plot(comp.index, comp.values, color=color, linewidth=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Sales')

plt.suptitle('Additive Decomposition (Period = 365 days)', fontsize=14, fontweight='bold')
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
dow_avg = ts.copy()
dow_avg['dow'] = dow_avg.index.day_name()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_means = dow_avg.groupby('dow')['total_sales'].mean().reindex(order)
bars = ax.bar(order, dow_means.values, color=sns.color_palette('muted', 7))
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.set_title('Average Sales by Day of Week (Cyclic Pattern)', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Total Sales')
plt.tight_layout()
plt.show()

## 5 · Stationarity Check

In [ ]:
series = ts['total_sales'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)
roll_mean = series.rolling(30).mean()
roll_var  = series.rolling(30).var()

axes[0].plot(series, color='steelblue', linewidth=0.7, label='Original')
axes[0].plot(roll_mean, color='tomato', linewidth=2, label='Rolling Mean (30d)')
axes[0].set_title('Rolling Mean', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].plot(roll_var, color='green', linewidth=1, label='Rolling Variance (30d)')
axes[1].set_title('Rolling Variance', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Stationarity Diagnostics', fontsize=14, fontweight='bold')
plt.show()

adf_result = adfuller(series, autolag='AIC')
adf_stat, p_value, n_lags, n_obs = adf_result[0], adf_result[1], adf_result[2], adf_result[3]
critical_values = adf_result[4]

is_stationary = p_value < 0.05

print('=' * 55)
print('       AUTOMATIC STATIONARITY REPORT')
print('=' * 55)
print(f'  ADF Statistic     : {adf_stat:.4f}')
print(f'  p-value           : {p_value:.6f}')
print(f'  Lags used         : {n_lags}')
print(f'  Observations used : {n_obs}')
print('-' * 55)
for key, val in critical_values.items():
    print(f'  Critical Value {key} : {val:.4f}')
print('-' * 55)
status = '✅ STATIONARY' if is_stationary else '⚠️  NON-STATIONARY'
print(f'  Verdict           : {status}')
if not is_stationary:
    print('  Recommendation    : Apply first differencing (d=1 in ARIMA)')
else:
    print('  Recommendation    : Series is ready for direct modelling')
print('=' * 55)

In [ ]:
d_order = 0 if is_stationary else 1

if d_order == 1:
    diff_series = series.diff().dropna()
    adf2 = adfuller(diff_series, autolag='AIC')
    print(f'ADF on differenced series — p-value: {adf2[1]:.6f}')
    verdict2 = 'Stationary ✅' if adf2[1] < 0.05 else 'Still non-stationary ⚠️'
    print(f'Verdict after first differencing: {verdict2}')
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(diff_series, color='teal', linewidth=0.8)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_title('First-Differenced Series', fontsize=12, fontweight='bold')
    ax.set_ylabel('Δ Sales')
    plt.tight_layout()
    plt.show()

## 6 · Forecasting Models — Train / Test Split

In [ ]:
TEST_DAYS = 90
train_s = series.iloc[:-TEST_DAYS]
test_s  = series.iloc[-TEST_DAYS:]

print(f'Training period : {train_s.index[0].date()} → {train_s.index[-1].date()}  ({len(train_s)} obs)')
print(f'Test period     : {test_s.index[0].date()}  → {test_s.index[-1].date()}   ({len(test_s)} obs)')

### 6a · ARIMA

In [ ]:
arima_order = (2, d_order, 2)
arima_model = ARIMA(train_s, order=arima_order, freq='D')
arima_fit   = arima_model.fit()

arima_forecast_obj = arima_fit.get_forecast(steps=TEST_DAYS)
arima_pred         = arima_forecast_obj.predicted_mean
arima_ci           = arima_forecast_obj.conf_int(alpha=0.05)

arima_pred.index = test_s.index
arima_ci.index   = test_s.index

print(arima_fit.summary().tables[0])
print(f'\nARIMA order : {arima_order}')

### 6b · Prophet

In [ ]:
if PROPHET_AVAILABLE:
    prophet_train = train_s.reset_index().rename(columns={'date': 'ds', 'total_sales': 'y'})
    prophet_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.95,
        changepoint_prior_scale=0.1
    )
    prophet_model.fit(prophet_train)

    future_test   = prophet_model.make_future_dataframe(periods=TEST_DAYS, freq='D')
    prophet_full  = prophet_model.predict(future_test)
    prophet_pred  = prophet_full.set_index('ds')['yhat'].loc[test_s.index]
    prophet_lower = prophet_full.set_index('ds')['yhat_lower'].loc[test_s.index]
    prophet_upper = prophet_full.set_index('ds')['yhat_upper'].loc[test_s.index]
    print('✅ Prophet trained successfully.')
else:
    print('⚠️  Prophet not available — skipping Prophet model.')

## 7 · Forecast Horizons — Future Predictions

In [ ]:
HORIZONS = {'7 days': 7, '30 days': 30, '90 days': 90, '6 months': 180}

arima_full_model = ARIMA(series, order=arima_order, freq='D').fit()

future_forecasts = {}
for label, horizon in HORIZONS.items():
    fc = arima_full_model.get_forecast(steps=horizon)
    future_forecasts[label] = {
        'mean'  : fc.predicted_mean,
        'lower' : fc.conf_int(alpha=0.05).iloc[:, 0],
        'upper' : fc.conf_int(alpha=0.05).iloc[:, 1],
    }

if PROPHET_AVAILABLE:
    prophet_full_train = series.reset_index().rename(columns={'date': 'ds', 'total_sales': 'y'})
    prophet_full_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.95,
        changepoint_prior_scale=0.1
    )
    prophet_full_model.fit(prophet_full_train)

prophet_future_forecasts = {}
if PROPHET_AVAILABLE:
    for label, horizon in HORIZONS.items():
        fut = prophet_full_model.make_future_dataframe(periods=horizon, freq='D')
        pfc = prophet_full_model.predict(fut).tail(horizon).set_index('ds')
        prophet_future_forecasts[label] = {
            'mean'  : pfc['yhat'],
            'lower' : pfc['yhat_lower'],
            'upper' : pfc['yhat_upper'],
        }

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
history_tail = series.tail(180)

for ax, (label, fc) in zip(axes.flat, future_forecasts.items()):
    ax.plot(history_tail.index, history_tail.values, color='steelblue', linewidth=1, label='Historical')
    ax.plot(fc['mean'].index, fc['mean'].values, color='tomato', linewidth=1.5, label='ARIMA Forecast')
    ax.fill_between(fc['mean'].index, fc['lower'].values, fc['upper'].values,
                    color='tomato', alpha=0.15, label='95% CI')
    if PROPHET_AVAILABLE and label in prophet_future_forecasts:
        pfc = prophet_future_forecasts[label]
        ax.plot(pfc['mean'].index, pfc['mean'].values, color='green', linewidth=1.5,
                linestyle='--', label='Prophet Forecast')
    ax.set_title(f'Forecast — {label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Total Sales')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('ARIMA vs Prophet — Multi-Horizon Forecasts', fontsize=14, fontweight='bold')
plt.show()

## 8 · Model Evaluation

In [ ]:
def mape(actual, predicted):
    actual_s, pred_s = np.array(actual), np.array(predicted)
    mask = actual_s != 0
    return np.mean(np.abs((actual_s[mask] - pred_s[mask]) / actual_s[mask])) * 100

def evaluate(name, actual, predicted):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mpe  = mape(actual, predicted)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mpe}

results = []
results.append(evaluate('ARIMA', test_s.values, arima_pred.values))

if PROPHET_AVAILABLE:
    results.append(evaluate('Prophet', test_s.values, prophet_pred.values))

metrics_df = pd.DataFrame(results).set_index('Model').round(2)
print('\n' + '='*45)
print('     MODEL EVALUATION METRICS (90-day test)')
print('='*45)
print(metrics_df.to_string())
print('='*45)
metrics_df

## 9 · Actual vs Predicted Plot with Confidence Intervals

In [ ]:
fig, axes = plt.subplots(2 if PROPHET_AVAILABLE else 1, 1,
                         figsize=(14, 10 if PROPHET_AVAILABLE else 5),
                         constrained_layout=True)

if not PROPHET_AVAILABLE:
    axes = [axes]

context_start = train_s.index[-90]
context = series.loc[context_start:train_s.index[-1]]

axes[0].plot(context.index, context.values, color='steelblue', linewidth=1.2, label='Historical (last 90d)')
axes[0].plot(test_s.index, test_s.values, color='black', linewidth=1.5, linestyle='--', label='Actual')
axes[0].plot(arima_pred.index, arima_pred.values, color='tomato', linewidth=2, label='ARIMA Predicted')
axes[0].fill_between(arima_ci.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1],
                     color='tomato', alpha=0.15, label='95% CI')
axes[0].axvline(test_s.index[0], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[0].set_title('ARIMA — Actual vs Predicted (90-day Test)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Sales')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

if PROPHET_AVAILABLE:
    axes[1].plot(context.index, context.values, color='steelblue', linewidth=1.2, label='Historical (last 90d)')
    axes[1].plot(test_s.index, test_s.values, color='black', linewidth=1.5, linestyle='--', label='Actual')
    axes[1].plot(prophet_pred.index, prophet_pred.values, color='green', linewidth=2, label='Prophet Predicted')
    axes[1].fill_between(prophet_pred.index, prophet_lower.values, prophet_upper.values,
                         color='green', alpha=0.15, label='95% CI')
    axes[1].axvline(test_s.index[0], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
    axes[1].set_title('Prophet — Actual vs Predicted (90-day Test)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Total Sales')
    axes[1].legend()
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Actual vs Predicted — With 95% Confidence Intervals', fontsize=14, fontweight='bold')
plt.show()

## 10 · Advanced Reporting — ARIMA vs Prophet Comparison

In [ ]:
comparison_data = {
    'Attribute'             : ['Model Type', 'Stationarity Required', 'Seasonality Handling',
                               'Holiday Support', 'Auto-Changepoints',
                               'Interpretability', 'Uncertainty Intervals',
                               'Training Speed', 'Best For'],
    'ARIMA'                 : ['Statistical', 'Yes (differencing)', 'Partial (SARIMA)',
                               'Manual regressors', 'No',
                               'High (coefficients)', 'Yes (via MLE)', 'Fast',
                               'Stationary / short series'],
    'Prophet'               : ['Decomposable', 'No', 'Additive/Multiplicative',
                               'Built-in', 'Yes',
                               'High (components)', 'Yes (MCMC / MAP)', 'Moderate',
                               'Seasonal + trend shifts'],
}
comparison_df = pd.DataFrame(comparison_data).set_index('Attribute')

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
tbl = ax.table(
    cellText=comparison_df.reset_index().values,
    colLabels=['Attribute', 'ARIMA', 'Prophet'],
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f0f4f8')
    if c == 0:
        cell.set_text_props(fontweight='bold')
ax.set_title('ARIMA vs Prophet — Feature Comparison', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print(comparison_df.to_string())

In [ ]:
if len(results) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
    metric_names = ['MAE', 'RMSE', 'MAPE (%)']
    colors = ['#e74c3c', '#27ae60']
    for ax, metric in zip(axes, metric_names):
        vals = [metrics_df.loc[m, metric] for m in metrics_df.index]
        bars = ax.bar(metrics_df.index, vals, color=colors[:len(vals)], edgecolor='white', linewidth=1.5)
        ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=10, fontweight='bold')
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.set_ylabel('Value')
    plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
    plt.show()
else:
    print('Only one model available — bar chart comparison skipped.')

### Forecast Summary Report

In [ ]:
best_model_name = metrics_df['MAPE (%)'].idxmin() if len(results) > 1 else results[0]['Model']
best_mape       = metrics_df.loc[best_model_name, 'MAPE (%)']

fc_6m = future_forecasts['6 months']['mean']
peak_date  = fc_6m.idxmax().date()
peak_val   = fc_6m.max()

trend_direction = 'Upward 📈' if fc_6m.iloc[-1] > fc_6m.iloc[0] else 'Downward 📉'

report = f"""
╔══════════════════════════════════════════════════════╗
║          FORECAST SUMMARY REPORT                     ║
╠══════════════════════════════════════════════════════╣
║  Dataset        : Corporación Favorita Grocery Sales ║
║  Target         : Aggregate Daily Sales (all stores) ║
║  Training end   : {train_s.index[-1].date()}                        ║
║  Forecast start : {fc_6m.index[0].date()}                        ║
║  Forecast end   : {fc_6m.index[-1].date()}                        ║
║  Best model     : {best_model_name:<37}║
║  MAPE           : {best_mape:.2f}%{' ' * (36 - len(f'{best_mape:.2f}%'))}║
║  Expected trend : {trend_direction:<37}║
║  Peak forecast  : {str(peak_date):<37}║
║  Peak value     : {peak_val:,.0f}{' ' * max(0, 37-len(f'{peak_val:,.0f}'))}║
╚══════════════════════════════════════════════════════╝
"""
print(report)

## 11 · Interactive Forecast Analysis — Historical vs Predicted vs Seasonal Effects

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 14), constrained_layout=True)

axes[0].plot(series.index, series.values, color='steelblue', linewidth=0.8, alpha=0.9, label='Historical')
axes[0].plot(series.rolling(30).mean(), color='tomato', linewidth=2, label='30d Trend')
axes[0].set_title('Historical Sales Trend', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Sales')
axes[0].legend()

best_fc   = future_forecasts['6 months']
axes[1].plot(series.tail(90).index, series.tail(90).values,
             color='steelblue', linewidth=1.2, label='Last 90d Actual')
axes[1].plot(best_fc['mean'].index, best_fc['mean'].values,
             color='tomato', linewidth=2, label='ARIMA Forecast (6m)')
axes[1].fill_between(best_fc['mean'].index, best_fc['lower'].values, best_fc['upper'].values,
                     color='tomato', alpha=0.15, label='95% CI')
if PROPHET_AVAILABLE and '6 months' in prophet_future_forecasts:
    pfc6 = prophet_future_forecasts['6 months']
    axes[1].plot(pfc6['mean'].index, pfc6['mean'].values,
                 color='green', linewidth=2, linestyle='--', label='Prophet Forecast (6m)')
axes[1].axvline(series.index[-1], color='gray', linewidth=1.5, linestyle=':', label='Today')
axes[1].set_title('6-Month Ahead Forecast — ARIMA vs Prophet', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Total Sales')
axes[1].legend()

seasonal_comp = decomp.seasonal
axes[2].fill_between(seasonal_comp.index, seasonal_comp.values, alpha=0.5,
                     color='purple', label='Seasonal Effect')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_title('Seasonal Effects (Additive Decomposition)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Seasonal Component')
axes[2].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Interactive Forecast Analysis Dashboard', fontsize=15, fontweight='bold')
plt.show()

## 12 · Standout Features Summary

In [ ]:
if PROPHET_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    prophet_components = prophet_full_model.predict(
        prophet_full_model.make_future_dataframe(periods=180, freq='D')
    )

    axes[0].plot(prophet_components['ds'], prophet_components['trend'], color='tomato', linewidth=1.5)
    axes[0].set_title('Prophet — Trend Component', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Trend')
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

    weekly_comp = prophet_components[['ds', 'weekly']].copy()
    weekly_comp['dow'] = pd.to_datetime(weekly_comp['ds']).dt.day_name()
    dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    weekly_mean = weekly_comp.groupby('dow')['weekly'].mean().reindex(dow_order)
    axes[1].bar(weekly_mean.index, weekly_mean.values, color=sns.color_palette('muted', 7))
    axes[1].set_title('Prophet — Weekly Seasonal Effect', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Weekly Component')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

    plt.suptitle('Prophet Component Analysis', fontsize=14, fontweight='bold')
    plt.show()
else:
    print('Prophet not available — component analysis skipped.')

In [ ]:
print('\n' + '='*60)
print('       FINAL FORECAST TABLE — ALL HORIZONS (ARIMA)')
print('='*60)

summary_rows = []
for label, horizon in HORIZONS.items():
    fc = future_forecasts[label]
    summary_rows.append({
        'Horizon'   : label,
        'Start'     : fc['mean'].index[0].date(),
        'End'       : fc['mean'].index[-1].date(),
        'Mean Fcst' : round(fc['mean'].mean(), 0),
        'Min Fcst'  : round(fc['mean'].min(), 0),
        'Max Fcst'  : round(fc['mean'].max(), 0),
        'CI Lower'  : round(fc['lower'].mean(), 0),
        'CI Upper'  : round(fc['upper'].mean(), 0),
    })

summary_table = pd.DataFrame(summary_rows).set_index('Horizon')
print(summary_table.to_string())
print('='*60)
summary_table

In [ ]:
print('\n✅ Notebook completed successfully.')
print(f'   Models trained    : {[r["Model"] for r in results]}')
print(f'   Forecast horizons : {list(HORIZONS.keys())}')
print(f'   Best model (MAPE) : {best_model_name} @ {best_mape:.2f}%')
print(f'   Peak forecast on  : {peak_date}')